# ContractRisk — diagnóstico visual 18

La prueba 17 terminó técnicamente, pero el OCR añadió un cero a un importe y el control bloqueó la respuesta. Este diagnóstico **post-hoc** tiene ocho imágenes ya congeladas en el repositorio: no cambia modelos, prompts ni guardas. No sustituye resultados anteriores ni constituye validación ciega.

Una sola pasada en **GPU T4**, conservando todas las salidas. La captura original se replica una vez para estudiar OCR; la pregunta de esta tanda solicita modalidad y valor, no la entidad de la corrida 17. Otros casos comparan formatos de dos fichas e incluyen tres controles. No hay API del juez ni gastos adicionales. No modifica main ni el Space.

Ejecutar en orden. Si falla preparación, ir directamente a la última celda para conservar evidencia; no ejecutar inferencia ni repetir por cuenta propia.

In [1]:
import os,sys,subprocess,tempfile,json
from pathlib import Path
os.environ['USE_TF']='0'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['ANONYMIZED_TELEMETRY']='false'
SETUP_OK=False
ROOT=Path(tempfile.mkdtemp(prefix='contractrisk_visual18_',dir='/content'))/'repo'
subprocess.run(['git','clone','--single-branch','--branch','Entrega/contractrisk-final','https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git',str(ROOT)],check=True)
os.chdir(ROOT)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('Código:',COMMIT)
def run_visible(arguments,logfile):
    with open(logfile,'w',encoding='utf-8') as log:
        process=subprocess.Popen(arguments,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        status=process.wait()
    if status:raise RuntimeError('Proceso detenido. Conservar diagnóstico con la última celda: '+logfile)


Código: ff951c72e08925fd3f47312e3a07b4c0e271017e


In [2]:
SETUP_OK=False
run_visible([sys.executable,'final/prepare_test_history.py'],'visual18_history.log')
run_visible([sys.executable,'-m','pip','install','-r','final/space_candidate/requirements.txt'],'visual18_install.log')
run_visible([sys.executable,'final/service_environment.py','--output','visual18_environment.json'],'visual18_environment.log')
run_visible([sys.executable,'-c',"import torch; assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'Selecciona T4'; print(torch.cuda.get_device_name(0))"],'visual18_gpu.log')
run_visible([sys.executable,'-m','unittest','discover','-s','final/tests','-v'],'visual18_tests_final.log')
run_visible([sys.executable,'-m','unittest','discover','-s','m3/tests','-v'],'visual18_tests_m3.log')
from final.visual_deployment18 import load_protocol
protocol=load_protocol()
print('Casos congelados:',[c['id'] for c in protocol['cases']])
SETUP_OK=True


{
  "source_commit": "ff951c72e08925fd3f47312e3a07b4c0e271017e",
  "was_shallow": false,
  "history_complete": true,
  "checkout_unchanged": true,
  "historical_commits_verified": [
    "6ec8dcdf22433c632b5eb0e5539401c33b9c923d",
    "71912f1c5755790381a8e39c8a67801bcbf9243d",
    "7afdda0212bf38ad1accd043a968db722569c618",
    "ece58275c2a19abfb6bbd2fe4b507731232e9083"
  ]
}
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 92.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 124.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 122

In [3]:
if not SETUP_OK:raise RuntimeError('Preparación incompleta. Descargar diagnóstico sin ejecutar inferencia.')
RESULTS=ROOT/'visual18_results'
run_visible([sys.executable,'-u','final/visual_deployment18.py','run','--output',str(RESULTS)],'visual18_inference.log')


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given

Fetching 7 files: 100%|██████████| 7/7 [00:07<00:00,  1.01s/it]
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at BSC-LT/RoBERTalex and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
`torch_dtype` is deprecated! Use `dtype` instead!

Fetching 2 files: 100%|██████████| 2/2 [01:02<00:00, 31.50s/it]

Loadi

## Lectura y evidencia

`completed_requires_review` significa que terminó la tanda, no que se aprobaron las respuestas o el despliegue. Los cinco casos legibles y tres controles se leen por separado. Una abstención por OCR errado no cuenta como lectura correcta. Los chequeos de importes/modalidad son literales, no revisión semántica humana.

Se conservan transcripciones, respuestas, fuentes, guardas y resultados parciales. El ZIP y el notebook con salidas permiten la revisión posterior. No publicar archivos en el Space ni repetir para buscar un mejor resultado.

In [4]:
import zipfile
from google.colab import files
metadata=ROOT/'visual18_results/metadata.json'
print(metadata.read_text(encoding='utf-8') if metadata.exists() else 'La inferencia no comenzó o no generó metadata; consultar logs.')
archive=ROOT.parent/'contractrisk_final_visual18_resultados.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in ['visual18_results','final/evidence/visual18_protocol']:
        base=ROOT/folder
        if base.exists():
            for item in sorted(base.rglob('*')):
                if item.is_file():z.write(item,item.relative_to(ROOT))
    for pattern in ['visual18_*.log','visual18_environment.json']:
        for item in sorted(ROOT.glob(pattern)):z.write(item,item.name)
    z.writestr('checkout_commit.txt',COMMIT+'\n')
    z.writestr('setup_state.json',json.dumps({'setup_ok':SETUP_OK}))
files.download(str(archive))


{
  "status": "completed_requires_review",
  "protocol_sha256": "cc1f3a9ac6b466d49a479a98029a0c6e2d73bfe570196071f641e25548112b3d",
  "source_commit": "ff951c72e08925fd3f47312e3a07b4c0e271017e",
  "purpose": "post_hoc_frozen_visual_diagnostic_not_final_benchmark",
  "api_calls": 0,
  "started_utc": "2026-11-20T00:30:31.346864+00:00",
  "prior_results_replaced": false,
  "environment": {
    "ready": true,
    "packages": {
      "gradio": "6.16.0",
      "transformers": "4.57.6",
      "sentence-transformers": "3.4.1",
      "chromadb": "0.6.3",
      "numpy": "2.1.3",
      "rank-bm25": "0.2.2",
      "sentencepiece": "0.2.1",
      "peft": "0.17.1",
      "accelerate": "1.10.1",
      "huggingface-hub": "0.36.2",
      "pillow": "11.3.0",
      "wandb": "0.31.0",
      "pysqlite3-binary": "0.5.4.post2",
      "torch": "2.14.1+cu130"
    },
    "missing": [],
    "mismatched": [],
    "conflicts": []
  },
  "gpu": "Tesla T4",
  "health_before": {
    "models": {
      "generator": tru

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>